In [ ]:
import astropy.units as u
import numpy as np
import pandas as pd

from ctlearn_manager import CTLearnTriModelManager, DataSample, load_model_from_index
from ctlearn_manager.utils import ClusterConfiguration, ParticleType

# 💾 Load models

In [2]:
def get_surrounding_nodes(index, nodes, num_surrounding=1):
    surrounding_indices = []
    for i in range(1, num_surrounding + 1):
        if index - i >= 0:
            surrounding_indices.append(index - i)
        if index + i < len(nodes):
            surrounding_indices.append(index + i)
        if len(surrounding_indices) >= num_surrounding:
            break
    return surrounding_indices


def angular_distance(ze1, az1, ze2, az2):
    ze1, az1, ze2, az2 = map(np.radians, [ze1, az1, ze2, az2])
    delta_az = az2 - az1
    delta_ze = ze2 - ze1
    a = (
        np.sin(delta_ze / 2) ** 2
        + np.cos(ze1) * np.cos(ze2) * np.sin(delta_az / 2) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return c


def get_testing_node(
    first_node_index,
    training_nodes_gammas,
    training_nodes_protons,
    testing_nodes_gammas,
):
    zes = [
        training_nodes_gammas["ze"][first_node_index],
        training_nodes_gammas["ze"][first_node_index + 1],
    ]
    azs = [
        training_nodes_gammas["az"][first_node_index],
        training_nodes_gammas["az"][first_node_index + 1],
    ]
    zes_protons = []
    azs_protons = []
    for index in [first_node_index, first_node_index + 1]:
        surrounding_indices = get_surrounding_nodes(index, training_nodes_protons["ze"])
        zes_protons.extend(training_nodes_protons["ze"][surrounding_indices])
        azs_protons.extend(training_nodes_protons["az"][surrounding_indices])
    same_ze_indices = training_nodes_protons[
        training_nodes_protons["ze"].isin(zes_protons)
    ].index
    zes_protons = training_nodes_protons["ze"][same_ze_indices]
    azs_protons = training_nodes_protons["az"][same_ze_indices]
    # Find the closest gamma point testing node for the two training gamma diffuse nodes
    min_distance = float("inf")
    closest_testing_node = None
    mid_ze = np.mean(zes)
    mid_az = np.mean(azs)

    distances = angular_distance(
        mid_ze, mid_az, testing_nodes_gammas["ze"], testing_nodes_gammas["az"]
    )
    closest_index = np.argmin(distances)
    closest_testing_node = (
        testing_nodes_gammas["ze"][closest_index],
        testing_nodes_gammas["az"][closest_index],
    )

    # Extract the zenith and azimuth angles of the closest testing node
    closest_ze_testing = [closest_testing_node[0]]
    closest_az_testing = [closest_testing_node[1]]
    return zes, azs, zes_protons, azs_protons, closest_ze_testing, closest_az_testing

In [ ]:
MODEL_INDEX_FILE = "/users/blacave/PhD/LST/ctlearn_models_index.h5"
training_nodes_gammas = pd.read_csv(
    "/users/blacave/PhD/LST/CTLearnManager/TrainingNodesGammaDiffuse.csv"
)
training_nodes_protons = pd.read_csv(
    "/users/blacave/PhD/LST/CTLearnManager/TrainingNodesProtonDiffuse.csv"
)
testing_nodes_gammas = pd.read_csv(
    "/users/blacave/PhD/LST/CTLearnManager/TestingNodesGammaPoint.csv"
)
testing_nodes_protons = pd.read_csv(
    "/users/blacave/PhD/LST/CTLearnManager/TestingNodesProton.csv"
)
cluster_congig = ClusterConfiguration(
    memory_mb=64000, use_cluster=True, time="08:00:00", nodes=1, account="cta08"
)
for i in [9]:
    energy_model = load_model_from_index(f"LST1_energy_CRABdec_{i}", MODEL_INDEX_FILE)
    direction_model = load_model_from_index(
        f"LST1_cameradirection_CRABdec_{i}", MODEL_INDEX_FILE
    )
    type_model = load_model_from_index(f"LST1_type_CRABdec_{i}", MODEL_INDEX_FILE)
    Stereo_Tri_Model = CTLearnTriModelManager(
        direction_model=direction_model,
        energy_model=energy_model,
        type_model=type_model,
        cluster_configuration=cluster_congig,
    )

    # zes, azs, zes_protons, azs_protons, closest_ze_testing, closest_az_testing = get_testing_node(i, training_nodes_gammas, training_nodes_protons, testing_nodes_gammas)
    testing_samples = []

    for ze, az in zip(testing_nodes_gammas["ze"], testing_nodes_gammas["az"]):
        # print(f"gamma_theta_{ze}_az_{az}_runs*.dl1.h5")
        testing_samples.append(
            DataSample(
                directory="/capstor/scratch/cscs/tmiener/datasets/LST1/test/gamma/",
                pattern=f"gamma_theta_{int(ze)}*_az_{int(az)}*_runs*.dl1.h5",
            )
        )
    for ze, az in zip(testing_nodes_protons["ze"], testing_nodes_protons["az"]):
        testing_samples.append(
            DataSample(
                directory="/capstor/scratch/cscs/tmiener/datasets/LST1/test/proton/",
                pattern=f"proton_theta_{int(ze)}*_az_{int(az)}*_runs*.dl1.h5",
            )
        )

    Stereo_Tri_Model.set_testing_data(testing_samples=testing_samples)

🧠 Model name: LST1_energy_CRABdec_0
🧠 Model name: LST1_direction_CRABdec_0
🧠 Model name: LST1_type_CRABdec_0
💾 Model LST1_direction_CRABdec_0 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_energy_CRABdec_0 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_type_CRABdec_0 testing data update:
	➡️ Testing gamma data updated
🧠 Model name: LST1_energy_CRABdec_1
🧠 Model name: LST1_direction_CRABdec_1
🧠 Model name: LST1_type_CRABdec_1
💾 Model LST1_direction_CRABdec_1 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_energy_CRABdec_1 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_type_CRABdec_1 testing data update:
	➡️ Testing gamma data updated
🧠 Model name: LST1_energy_CRABdec_2
🧠 Model name: LST1_direction_CRABdec_2
🧠 Model name: LST1_type_CRABdec_2
💾 Model LST1_direction_CRABdec_2 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_energy_CRABdec_2 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1

# 🗃️ Set testing files and directions

In [4]:
# Stereo_Tri_Model.set_testing_directories(
#     testing_gamma_dirs = ["/home/blacave/CTLearn/Data/DL1/SST1M/MC/Gamma_point/20deg/testing/"],
#     testing_proton_dirs = ["/home/blacave/CTLearn/Data/DL1/SST1M/MC/Proton_diffuse/20deg/merged/testing/"],
#     testing_gamma_zenith_distances = [20],
#     testing_gamma_azimuths = [0],
#     testing_proton_zenith_distances = [20],
#     testing_proton_azimuths = [0]
#     )

# 🧪 Launch testing

In [ ]:
Stereo_Tri_Model.get_available_testing_directions()
Stereo_Tri_Model.cluster_configuration.info()

(ZD, Az): (37.814, 90.0)


In [ ]:
Stereo_Tri_Model.launch_testing(
    32.059 * u.deg,
    248.099 * u.deg,
    ["/capstor/scratch/cscs/blacave/Testing_models/9/"],
    launch_particle_types=[ParticleType.GAMMA_POINT],
    config_dir="/capstor/scratch/cscs/blacave/Testing_models/9/",
    batch_size=512,
)

In [ ]:
Stereo_Tri_Model.merge_DL2_files(
    20,
    0,
    "/home/blacave/CTLearn/Data/DL2/Testing/merged/gamma_point_50_300E3GeV_20_20deg.h5",
    "/home/blacave/CTLearn/Data/DL2/Testing/merged/proton_diffuse_400_500E3GeV_20_20deg.h5",
    overwrite=True,
)